# EGMS Check — pobranie danych EGMS dla stacji PTO1 (przypadek 1)
Produkt: **EGMS Calibrated (L2B)**, ścieżki 117 i 44 (wstępujące) oraz 95 (zstępująca), pobierany przez oficjalne API.

Kolejność (nie pomijaj kroków):
1. Dysk Google i kod; dane trafiają na Dysk: `MyDrive/GeoWorldLook/egmscheck/data/01_Raw_EGMS/`.
2. Token: klucz serwisowy CLMS na Dysku **poza repozytorium** (`MyDrive/GeoWorldLook/secrets/clms_token.json`).
   Token nie jest drukowany ani zapisywany.
3. Listy wartości API: które wydania są dostępne (pytanie A2) — zapis do `data/00_Metadata/`.
4. Wyszukanie plików dla punktu PTO1 we wszystkich wydaniach: nazwy, bursty, rozmiary. **Nic się nie pobiera.**
5. **Stop:** decyzja o wydaniu (D-A) i zgoda na rozmiar. Dopiero potem krok 6 z listą plików.
6. Pobranie przyrostowe z SHA-256 i wpisem w `MANIFEST.csv`.
7. Sprawdzenie, czy Nevada Geodetic Laboratory ma stację PTO1 (dłuższy szereg GNSS).

## 1. Dysk Google i kod

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/geoworldlook/egms-check.git'
BRANCH = 'main'
PROJECT_DIR = '/content/drive/MyDrive/GeoWorldLook/egmscheck'
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)
    !git clone -b "{BRANCH}" "{REPO_URL}" "{PROJECT_DIR}"
else:
    !git -C "{PROJECT_DIR}" fetch origin "{BRANCH}"
    !git -C "{PROJECT_DIR}" checkout "{BRANCH}"
    !git -C "{PROJECT_DIR}" pull --ff-only origin "{BRANCH}"
!git -C "{PROJECT_DIR}" log -1 --format="Kod: %h %ci %s"
!pip install -q pyjwt pyproj
os.chdir(PROJECT_DIR)
for p in (PROJECT_DIR, os.path.join(PROJECT_DIR, 'prototypes')):
    if p not in sys.path:
        sys.path.insert(0, p)
# Po git pull usuń z pamięci moduły projektu wczytane wcześniej w tej sesji, żeby import wziął nowy kod
# (rozszerzenie autoreload w Colab z Pythonem 3.13 nie działa: importuje usunięty moduł `imp`).
for _m in ('step_01_ingest', 'egms_simple_check'):
    sys.modules.pop(_m, None)
print('Projekt:', PROJECT_DIR)

## 2. Stacja i token

In [ ]:
from pathlib import Path
import requests
from step_01_ingest import API_ENDPOINT, get_access_token, api_headers, get_api_lists, search_egms, point_query, hits_table
from egms_simple_check import read_pangaea_gnss

TOKEN_PATH = Path('/content/drive/MyDrive/GeoWorldLook/secrets/clms_token.json')   # klucz serwisowy CLMS (JSON)
gnss, STATION_LON, STATION_LAT = read_pangaea_gnss('data/02_Raw_Ground/1_gnss_pangaea/PTO1.txt')
print(f'PTO1 z pliku PANGAEA: lon {STATION_LON}, lat {STATION_LAT}')

session = requests.Session()
headers = api_headers(get_access_token(TOKEN_PATH, session))   # token nie jest drukowany

## 3. Listy wartości API (które wydania są dostępne?)

In [ ]:
import json, datetime
TODAY = datetime.date.today().strftime('%Y%m%d')
lists = get_api_lists(session, headers, API_ENDPOINT)
out = Path(f'data/00_Metadata/egms_api_lists_{TODAY}.json')
out.write_text(json.dumps(lists, indent=1, ensure_ascii=False))
for name, values in lists.items():
    shown = values if name != 'relative_orbits' else f'{len(values)} wartości, np. {values[:10]}'
    print(f'{name}: {shown}')
print('Zapisano', out)

## 4. Wyszukanie plików L2B dla punktu PTO1 we wszystkich wydaniach (bez pobierania)
Artykuł Fabris i Nazari 2026 podaje „sceny” 264–266 (ścieżka 117), 265–267 (44) i 806–808 (95);
porównaj je z kolumną `burstId`. Dwa trafienia na jednej ścieżce = stacja w zakładce burstów (duplikaty punktów).

In [ ]:
import pandas as pd
TRACKS = {117: 'ascending', 44: 'ascending', 95: 'descending'}     # Fabris i Nazari 2026, tab. 1
releases = lists['releases']
if not all(isinstance(r, str) for r in releases):
    raise TypeError(f'Nieoczekiwany format listy wydań: {releases}')
tables = []
for rel in releases:
    res = search_egms(session, headers, API_ENDPOINT, point_query(STATION_LON, STATION_LAT, 'L2B', rel))
    t = hits_table(res)
    t['query_release'] = rel
    tables.append(t)
hits = pd.concat(tables, ignore_index=True)
hits['relativeOrbit'] = hits['relativeOrbit'].astype(int)
hits_pto1 = hits[hits['relativeOrbit'].isin(TRACKS)]
cols = ['query_release', 'release', 'relativeOrbit', 'direction', 'burstId', 'filename', 'filesize_mb']
print(hits_pto1[cols].sort_values(['query_release', 'relativeOrbit', 'burstId']).to_string(index=False))
print()
print(hits_pto1.groupby('query_release')['filesize_mb'].agg(['count', 'sum']).rename(columns={'sum': 'total_MB'}))
other = sorted(set(hits['relativeOrbit']) - set(TRACKS))
print('Inne ścieżki nad stacją (nieużywane w artykule):', other)
csv = Path(f'data/00_Metadata/egms_search_PTO1_{TODAY}.csv')
hits.drop(columns=[c for c in ('poly', 'bbox') if c in hits.columns]).to_csv(csv, index=False)
print('Zapisano', csv)

## 5. STOP — decyzja przed pobraniem
Prześlij do Claude Code wynik kroków 3 i 4 (lista wydań, tabela, łączny rozmiar). Rozstrzygamy:
- wydanie (D-A, `docs/plans/Plan_v1_wdrozenie.md`): 2018–2022 i 2015–2021 są w archiwum CLMS; jeśli API ich nie zwraca,
  zostaje wniosek do CLMS Helpdesk albo wydanie 2019–2023 z jawnym odstępstwem od progu 85%,
- lista plików do pobrania (trzy ścieżki, bursty pokrywające stację).

## 6. Pobranie (po decyzji): wpisz wydanie i nazwy plików z tabeli wyżej

In [ ]:
from step_01_ingest import sync_egms
RELEASE = None              # np. '2019-2023' - zapis dokładnie jak w liście z kroku 3
CONFIRMED_FILES = []        # nazwy plików z kolumny filename, zaakceptowane po obejrzeniu rozmiarów
DOWNLOADED_BY = None        # kto pobiera (do manifestu)
PAUSE_S = 15                # przerwa między plikami; serwer odrzuca zbyt częste zapytania

if not CONFIRMED_FILES:
    print('Pominięte: brak potwierdzonej listy plików (krok 5).')
else:
    if RELEASE is None or DOWNLOADED_BY is None:
        raise ValueError('Uzupełnij RELEASE i DOWNLOADED_BY.')
    res = search_egms(session, headers, API_ENDPOINT, point_query(STATION_LON, STATION_LAT, 'L2B', RELEASE))
    report = sync_egms(session, headers, API_ENDPOINT, res, CONFIRMED_FILES,
                       Path('data/01_Raw_EGMS') / RELEASE / 'L2b', Path('data/00_Metadata/MANIFEST.csv'),
                       Path(PROJECT_DIR), PAUSE_S, DOWNLOADED_BY)
    print(report.to_string(index=False))

## 7. Czy Nevada Geodetic Laboratory ma stację PTO1?
Szereg PANGAEA kończy się 2021-04-13. Dłuższy szereg (E, N, U) z NGL rozwiązałby problem pokrycia 85% (D-A).
Ta komórka tylko sprawdza, czy strona stacji istnieje; niczego nie pobiera.

In [ ]:
url = 'https://geodesy.unr.edu/NGLStationPages/stations/PTO1.sta'
r = requests.get(url, timeout=60)
print(url, '->', r.status_code)
if r.ok:
    print(r.text[:1500])

## 8. NGL: pliki szeregu PTO1 i ich zakres dat (bez zapisu na Dysk)
Komórka znajduje na stronie stacji odnośniki do szeregów `tenv3`, wczytuje je **do pamięci** i podaje:
pierwszą i ostatnią datę, liczbę dób, przerwy > 30 dni. Nic nie zapisuje; zapis z SHA-256 i manifestem
dopiero po decyzji.

In [ ]:
import re, io
from urllib.parse import urljoin
import numpy as np
import pandas as pd
import requests
url = 'https://geodesy.unr.edu/NGLStationPages/stations/PTO1.sta'
page = requests.get(url, timeout=60)
page.raise_for_status()
links = sorted({urljoin(url, h) for h in re.findall(r'href=["\']([^"\']+)["\']', page.text) if h.lower().endswith('.tenv3')})   # bez README i innych plików
print('Odnośniki tenv3 na stronie stacji:')
for l in links:
    print('  ', l)
if not links:
    print('Brak odnośników tenv3 - wklej do Claude Code pełny tekst strony stacji.')
for l in links:
    r = requests.get(l, timeout=120)
    print('\n', l, '->', r.status_code)
    if not r.ok:
        continue
    t = pd.read_csv(io.StringIO(r.text), sep=r'\s+')
    if 'YYMMMDD' not in t.columns:
        print('Nieoczekiwane kolumny:', list(t.columns)[:20])
        continue
    dates = pd.to_datetime(t['YYMMMDD'], format='%y%b%d')
    gaps = dates.diff().dt.days
    print('kolumny:', list(t.columns)[:13])
    print('od', dates.min().date(), 'do', dates.max().date(), '| doby:', len(t))
    print('przerwy > 30 dni:', [(str(dates.iloc[i-1].date()), str(dates.iloc[i].date()), int(gaps.iloc[i])) for i in np.where(gaps > 30)[0]])

## 9. Inne stacje GNSS: przegląd NGL (MIDAS, IGS20) i pokrycie EGMS
Nie musimy ograniczać się do PTO1. NGL publikuje otwarte szeregi dobowe i prędkości MIDAS tysięcy stacji.
Aktualny plik prędkości w układzie IGS20 to `midas.IGS.txt` (katalog https://geodesy.unr.edu/velocities/);
`midas.IGS14.txt` nie jest już aktualizowany. Dane IGS20 sięgają na razie ok. 2024.65 (sierpień 2024).

Kryteria ustalamy **przed** obejrzeniem listy i bierzemy wszystkie stacje, które je spełniają (bez wyboru pod wynik).
Wartości poniżej to propozycja; zmień je przed uruchomieniem, jeśli chcesz inne.
EGMS kalibruje model GNSS m.in. na danych NGL, więc stacje z listy mogą nie być w pełni niezależne
(White Paper dopuszcza je do walidacji; trzeba to napisać wprost).

In [ ]:
import datetime
from pathlib import Path
import pandas as pd
from step_01_ingest import API_ENDPOINT, ngl_screening

# Kryteria (propozycja; zatwierdza autor przed obejrzeniem listy):
BBOX_LONLAT = (-25.0, 34.0, 45.0, 72.0)   # przybliżony zasięg EGMS (Europa)
T_FIRST_MAX = 2019.0                       # dane od początku wydania 2019-2023
T_LAST_MIN = 2024.25                       # 2019-2023 w całości i >= 85% wydania 2020-2024 (2020.0 + 0,85 * 5 lat);
                                           # dane NGL IGS20 sięgają na razie ok. 2024.65
VU_MAX_MM_YR = -2.0                        # osiadanie co najmniej 2 mm/rok
MAX_SEARCH = 30                            # dla ilu stacji pytać API EGMS o pokrycie
RELEASE_FOR_COVERAGE = '2019-2023'
MIDAS_URL = 'https://geodesy.unr.edu/velocities/midas.IGS.txt'   # IGS20; midas.IGS14.txt nie jest aktualizowany

cand_cov = ngl_screening(session, headers, API_ENDPOINT, MIDAS_URL, BBOX_LONLAT, T_FIRST_MAX, T_LAST_MIN,
                         VU_MAX_MM_YR, MAX_SEARCH, RELEASE_FOR_COVERAGE)
pd.set_option('display.width', 220)
print(cand_cov.to_string(index=False))
out = Path(f"data/00_Metadata/ngl_screening_{datetime.date.today():%Y%m%d}.csv")
cand_cov.to_csv(out, index=False)
print('Zapisano', out)